# Example Plots and Processing of Pipeline Analysis

In [1]:
# Import functions from the pipeline_fxn_lib.py script
import sys
sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/start_here/')
import pipeline_fxn_lib as lib

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.patches import Rectangle
import xarray as xr
import yaml

from statsmodels.nonparametric.smoothers_lowess import lowess
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

In [2]:
def removeLow(arr, cutoff = 1e-3):
    func = lambda x: (x > cutoff) * x
    vfunc = np.vectorize(func)
    return vfunc(arr)

In [3]:
# Define a consistent color scheme for GRUAN and ERA5 data
color_gruan = 'tab:green'
color_era5 = 'tab:blue'
color_gaussian = 'blue'  # For initial condition Gaussian, if needed

# Example usage:
# plt.plot(x_gruan, y_gruan, color=color_gruan, label='GRUAN')
# plt.plot(x_era5, y_era5, color=color_era5, label='ERA5')

## Downloaded and Processed ERA5 and GRUAN Data

## Process and Plot RHi Data

### Plot RHi Histogram

In [ ]:
mean, std_dev, filtered_gruan_rhi, filtered_era5_rhi = lib.post_process_met_data()

Combined meteorlogical data file already exists. Skipping combination.


In [ ]:
counts, xedges, yedges = np.histogram2d(filtered_gruan_rhi, filtered_era5_rhi, bins=50)

In [ ]:
plt.figure(figsize=(12,10), dpi=300)

# Mask zero values in counts to avoid log(0)
# masked_counts = np.ma.masked_where(counts == 0, counts)
masked_counts = np.ma.masked_array(counts, mask=(counts == 0))

# Set up LogNorm with safe min/max (log can't handle 0)
norm = LogNorm(vmin=masked_counts.min(), vmax=masked_counts.max())

# Display using imshow with LogNorm
im = plt.imshow(
    masked_counts.T,
    origin='lower',
    aspect='auto',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    cmap='Wistia',
    norm=norm
)

plt.xlabel('GRUAN RHi', fontsize=14)
plt.ylabel('ERA5 RHi', fontsize=14)
plt.ylim(bottom=0, top=180)
plt.xlim(left=0, right=180)
plt.title('2D Histogram of RHi: GRUAN vs ERA5', fontsize=16)
plt.plot([0,max(filtered_gruan_rhi)],[0,max(filtered_gruan_rhi)], 'r--', label='1:1')
plt.legend()

# Add colorbar with correct scaling
cbar = plt.colorbar(im, label='Number of Coincident Data Points (log scale)')


In [ ]:
df_combined = pd.read_parquet('/home/chinahg/GCresearch/contrailuncertainty/start_here/generated_files/combined_data.parquet')

In [ ]:
# Fit the ERA5 data to a Gaussian distribution, but only for RHi values between 90 and 140
era5_rhi_input = filtered_era5_rhi[(filtered_era5_rhi >= 110) & (filtered_era5_rhi <= 140)]
mirror_data = 2*3 - era5_rhi_input + 214
# Combine the original and mirrored data
combined_data = np.concatenate((era5_rhi_input, mirror_data))
# Fit a Gaussian distribution to the combined data
mean = np.mean(combined_data)
std_dev = np.std(combined_data)
print(f"Mean: {mean}, Standard Deviation: {std_dev}")

gaussian_samples = np.random.normal(mean, std_dev, 10000) # This is how we will sample the initial conditions

### Plot GRUAN vs ERA 5 histogram

In [ ]:
gruan_max = max(filtered_gruan_rhi)

plt.figure(figsize=(10,6), dpi=150)
plt.hist(filtered_gruan_rhi, bins=50, alpha=0.6, label='GRUAN RHi', color=color_gruan, edgecolor='black')
plt.hist(filtered_era5_rhi, bins=50, alpha=0.6, label='ERA5 RHi', color=color_era5, edgecolor='black')
plt.xlabel('RHi [%]')
plt.ylabel('Frequency')
plt.title('Histogram of RHi for GRUAN and ERA5')
plt.legend(loc='upper right', framealpha=1)
plt.xlim(0, gruan_max)
plt.tight_layout()

# Add magnified inset for 80 < RHi < gruan_max
ax = plt.gca()
axins = inset_axes(ax, width="40%", height="40%", loc='upper center', borderpad=2)
axins.hist(filtered_gruan_rhi, bins=50, alpha=0.6, color=color_gruan, edgecolor='black')
axins.hist(filtered_era5_rhi, bins=50, alpha=0.6, color=color_era5, edgecolor='black')
axins.set_xlim(80, 140)
axins.set_ylim(0,2000000)
axins.set_xticks(np.linspace(80, 140, 4, dtype = int))
axins.set_yticks(np.linspace(0, 2000000, 5, dtype = int))
axins.set_title('Zoom: 80 < RHi < 140', fontsize=10)

# Draw a box on the main plot to show the zoomed region
rect = Rectangle((80, 0), 140-80, ax.get_ylim()[1], linewidth=1, edgecolor='0.8', facecolor='none')
ax.add_patch(rect)
mark_inset(ax, axins, loc1=3, loc2=4, fc="yellow", ec="0", alpha = 0.2)

plt.show()

### Plot the absolute percent difference between ERA5 and GRUAN co-located RHi values

In [ ]:
# Compute the difference between ERA5 and GRUAN RHi values
rhi_diff = np.array(filtered_era5_rhi) - np.array(filtered_gruan_rhi)

plt.figure(figsize=(10, 7), dpi=150)
plt.scatter(filtered_era5_rhi, rhi_diff, alpha=0.3, s=0.5, color='orange')
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.xlabel('ERA5 RHi [%]')
plt.ylabel('Error [%]')
plt.title('Absolute Error of ERA5 RHi: |ERA5 - GRUAN| [%]')
plt.tight_layout()
plt.show()

In [ ]:
# Compute the difference between ERA5 and GRUAN RHi values
rhi_diff = abs(np.array(filtered_era5_rhi) - np.array(filtered_gruan_rhi))

# Group by unique RHi values and compute the average rhi_diff for each unique value

# Convert to pandas Series for grouping
era5_rhi_series = pd.Series(filtered_era5_rhi)
rhi_diff_series = pd.Series(rhi_diff)

# Group by unique ERA5 RHi values and compute the mean of rhi_diff for each
avg_diff_by_rhi = rhi_diff_series.groupby(era5_rhi_series).mean()

# Prepare x and y data from the avg_diff_by_rhi Series
x = avg_diff_by_rhi.index.values
y = avg_diff_by_rhi.values

# Plot the LOWESS smoothed curve
plt.figure(figsize=(10, 7), dpi=150)
plt.scatter(x, y, alpha=1, s=0.5, color='orange')
plt.xlabel('ERA5 RHi [%]')
plt.ylabel('Average Absolute Difference [%]')
plt.title('Absolute Difference between ERA5 and GRUAN RHi vs ERA5 RHi Samples')

In [ ]:
bin_width = 1
bins = np.arange(x.min(), x.max() + bin_width, bin_width)
bin_labels = (bins[:-1] + bins[1:]) / 2

# Bin the x values
binned = pd.cut(x, bins, labels=bin_labels, include_lowest=True)
binned_means = pd.Series(y).groupby(binned).mean()

plt.figure(figsize=(10, 6))
plt.plot(bin_labels, binned_means, marker='o', linestyle='-', color='orange')
plt.xlabel('ERA5 RHi [%]')
plt.ylabel('Average Absolute Difference [%]')
plt.title('Binned Moving Average of Absolute Difference vs ERA5 RHi')
plt.tight_layout()
plt.show()

### Create Gaussians based on initial conditions

In [ ]:
input_rhi_matrix = np.load("/home/chinahg/GCresearch/contrailuncertainty/start_here/generated_files/results/test_17/novel_samples_matrix.npy")

In [ ]:
print("Input RHi Matrix Shape:", input_rhi_matrix.shape)

In [ ]:
std = np.std(input_rhi_matrix[1,:])
mean = np.mean(input_rhi_matrix[1,0])
initial_condition_RHi = input_rhi_matrix[1,0]

x_gaussian = np.linspace(mean - 3*std, mean + 3*std, 1000)
y_gaussian = np.exp(-((x_gaussian-mean)/std)**2 / 2) / np.sqrt(2 * np.pi)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
plt.plot(x_gaussian, y_gaussian, color='orange', linewidth=2)
plt.axvline(initial_condition_RHi, color='0.5', linestyle='--', linewidth=1)

# Add ±1, ±2, ±3 std markers and shaded regions
for n, color, alpha in zip([1, 2, 3], ['#a6cee3', '#1f78b4', '#b2df8a'], [0.25, 0.15, 0.08]):
    left = initial_condition_RHi - n*std
    right = initial_condition_RHi + n*std
    plt.axvline(left, color=color, linestyle=':', linewidth=1)
    plt.axvline(right, color=color, linestyle=':', linewidth=1)
    plt.fill_between(x_gaussian, 0, y_gaussian, where=(x_gaussian >= left) & (x_gaussian <= right), color=color, alpha=alpha, label=f'±{n}σ')

# Add text annotation for standard error and standard deviation
# plt.text(
#     0.98, 0.95,
#     f'Reported Error = $\pm${IC_error:.2f}%\nStandard Deviation, σ = {std:.2f}%',
#     transform=ax.transAxes,
#     fontsize=12,
#     ha='right',
#     va='top',
#     bbox=dict(facecolor='white', alpha=0.7, edgecolor='none')
# )

# plt.ylim(0, 0.04)
plt.legend(loc='upper left')
plt.xlabel('RHi')
plt.ylabel('Probability Density')
plt.title('Initial Condition Error Informed Uncertainty')

In [ ]:
# Sample initial conditions
initial_conditions = np.linspace(20, 150, 3000)

# For each initial condition, compute the corresponding error and Gaussian
rhi_grid = []
pdf_grid = []

for ic in initial_conditions:
    idx = (np.abs(x - ic)).argmin()
    IC_error = y[idx] / 100 * ic
    std = IC_error / 3
    x_gaussian = np.linspace(0,200, 200)
    y_gaussian_ic = (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_gaussian - ic) / std) ** 2)
    rhi_grid.append(x_gaussian)
    pdf_grid.append(y_gaussian_ic)

# Convert to arrays for meshgrid
rhi_grid = np.array(rhi_grid)
initial_conditions_grid = np.array([initial_conditions]*rhi_grid.shape[1]).T
pdf_grid = np.array(pdf_grid)

fig = plt.figure(figsize=(12, 8), dpi=150)
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
surf = ax.plot_surface(initial_conditions_grid, rhi_grid, pdf_grid, cmap='viridis', alpha=0.85, edgecolor='none')

ax.set_xlabel('Initial Condition (RHi)')
ax.set_ylabel('RHi')
ax.set_zlabel('Probability Density')
ax.set_ylim(0,200)
ax.set_zlim(0, 0.8)
ax.set_title('3D Surface of Sampled Initial Conditions and Their Uncertainties')
fig.colorbar(surf, shrink=0.5, aspect=10, label='Probability Density')
ax.view_init(elev=20, azim=-30, roll=0)
plt.tight_layout()
plt.show()


## Plot Outputs

In [ ]:
# Import the data from the generated file
test_num = 30
test_id = f"test_{test_num}"
gen_data_path = f'/home/chinahg/GCresearch/contrailuncertainty/start_here/generated_files/results/test_{test_num}'

####################################################################################################################
### Test specifications for test number

test_specifications_path = f"{gen_data_path}/test_specifications.yaml"
# Load in the test specifications object
with open(test_specifications_path, "r") as f:
    spec_dict = yaml.unsafe_load(f)

# Reconstruct the APCEMMConfig object using the loaded dictionary
test_specifications = lib.APCEMMConfig()

# Update the object with the loaded parameters
for key, value in spec_dict.items():
    setattr(test_specifications, key, value)
####################################################################################################################
### Load in generated PCE and APCEMM data

# Load .npy files into variables
coefficients = np.load(f"{gen_data_path}/PCE_coefficients.npy")
alpha = np.load(f"{gen_data_path}/PCE_alpha_set.npy")

# Load in training inputs and outputs, validation inputs and outputs, and novel inputs and outputs
training_data =  np.load(f"{gen_data_path}/training_inputs_outputs_matrix.npy") # Num Runs x Timesteps x Variables
training_inputs = training_data[0,:,:] # Num Runs x Timesteps
true_training_solutions = training_data[1,:,:] # Num Runs x Timesteps

validation_data = np.load(f"{gen_data_path}/validation_inputs_outputs_matrix.npy") # Num Runs x Timesteps x Variables
validation_inputs = validation_data[0,:,:] # Num Runs x Timesteps
predicted_validation_solutions = np.load(f"{gen_data_path}/PCE_validation_solutions.npy") # Num Runs x Timesteps
true_validation_solutions = validation_data[1,:,:] # Num Runs x Timesteps

novel_inputs = np.load(f"{gen_data_path}/novel_input_matrix.npy") # Num Runs x Timesteps
novel_solutions = np.load(f"{gen_data_path}/novel_solutions.npy") # Num Runs x Timesteps
####################################################################################################################
### General constants for plotting and analysis

timesteps = len(training_data[0,0,:])
training_runs = test_specifications.training_runs
validation_runs = test_specifications.validation_runs
novel_runs = test_specifications.novel_runs

## Plot Training Data

In [ ]:
plt.figure(figsize=(10, 6), dpi=150)
lines = []
labels = []
rhi_values = []

num_lines = training_runs
cmap = cm.get_cmap('tab20', num_lines)

line_idx = 0
for idx in range(training_runs):
    color = cmap(line_idx)
    line, = plt.plot(
        np.arange(timesteps) * 10,  # Multiply x axis by 10
        true_training_solutions[idx, :],
        label=f"RHi = {training_inputs[idx,0]:.2f}",
        lw=1.5,
        color=color
    )
    lines.append(line)
    labels.append(f"RHi = {training_inputs[idx,0]:.2f}")
    rhi_values.append(training_inputs[idx,0])
    line_idx += 1

sorted_indices = np.argsort(rhi_values)
sorted_lines = [lines[i] for i in sorted_indices]
sorted_labels = [labels[i] for i in sorted_indices]

plt.xlabel("Time [min]")
plt.ylabel("Integrated VOD [m]")
plt.title(f"APCEMM Predicted Vertical Optical Depth for Mean RHi = 123% and Predicted Uncertainty = {test_specifications.IC_std_rhi*3:.2f}%")
plt.legend(sorted_lines, sorted_labels, fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# Compute the maximum error (max - min) at each timestep and normalize by the minimum value at that timestep
vod_max = np.max(true_training_solutions, axis=0)
vod_min = np.min(true_training_solutions, axis=0)
vod_error = np.abs(vod_max - vod_min)
relative_error = vod_error / vod_min
max_relative_error = np.max(relative_error)
print(f"Maximum relative error (max-min)/min across all timesteps: {max_relative_error:.2f}")

In [ ]:
# Compute the absolute error (max - min) at each timestep across all runs
vod_max = np.max(true_training_solutions, axis=0)
vod_min = np.min(true_training_solutions, axis=0)
vod_error = np.abs(vod_max - vod_min)

fig, ax1 = plt.subplots(figsize=(10, 6), dpi=150)

# Set common y-limits for both axes
ymin = np.min(true_training_solutions)
ymax = np.max(true_training_solutions)
ax1.set_ylim(ymin, ymax)

# Plot absolute error
ax1.plot(np.arange(timesteps) * 10, vod_error, color='red', lw=2, label='Absolute Error (Max - Min)')
ax1.set_xlabel("Time [min]")
ax1.set_ylabel("Absolute Integrated VOD Error (Max - Min) [m]", color='red')
ax1.tick_params(axis='y', labelcolor='red')

# Create a second y-axis for the individual runs
ax2 = ax1.twinx()
ax2.set_ylim(ymin, ymax)
for idx in range(training_runs):
    ax2.plot(np.arange(timesteps) * 10, true_training_solutions[idx, :], color='gray', alpha=0.4, lw=1)
ax2.set_ylabel("Integrated VOD [m]", color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

fig.suptitle("Absolute Error and All Training Runs: Integrated VOD")
fig.tight_layout()
plt.show()


In [ ]:
# Plot only the runs with RHi values closest to those in input_rhis
input_rhis = [140, 130, 120, 110, 100]
closest_indices = [np.abs(np.array(rhi_values) - rhi).argmin() for rhi in input_rhis]
print("Closest indices for input RHi values:", closest_indices)

plt.figure(figsize=(10, 6), dpi=150)
lines = []
labels = []

for idx in closest_indices:
    line, = plt.plot(
        np.arange(timesteps) * 10,
        true_training_solutions[idx, :],
        label=f"RHi = {training_inputs[idx,0]:.2f}",
        lw=1.5
    )
    lines.append(line)
    labels.append(f"RHi = {training_inputs[idx,0]:.2f}")

plt.xlabel("Time [min]")
plt.ylabel("Integrated VOD [m]")
plt.title(f"APCEMM Predicted Vertical Optical Depth for Mean RHi = 123% and Predicted Uncertainty = {test_specifications.IC_std_rhi*3:.2f}%")
plt.legend(lines, labels, fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# Compute the absolute error (max - min) at each timestep for the selected runs (closest_indices)
vod_max_sel = np.max(true_training_solutions[closest_indices, :], axis=0)
vod_min_sel = np.min(true_training_solutions[closest_indices, :], axis=0)
vod_error_sel = np.abs(vod_max_sel - vod_min_sel)

fig, ax1 = plt.subplots(figsize=(10, 6), dpi=150)

# Set common y-limits for both axes
ymin_sel = np.min(true_training_solutions[closest_indices, :])
ymax_sel = np.max(true_training_solutions[closest_indices, :])
ax1.set_ylim(ymin_sel, ymax_sel)

# Plot absolute error for selected runs
ax1.plot(np.arange(timesteps) * 10, vod_error_sel, color='red', lw=2, label='Absolute Error (Max - Min)')
ax1.set_xlabel("Time [min]")
ax1.set_ylabel("Absolute Integrated VOD Difference [m]", color='red')
ax1.tick_params(axis='y', labelcolor='red')

# Create a second y-axis for the individual selected runs
ax2 = ax1.twinx()
ax2.set_ylim(ymin_sel, ymax_sel)
for idx in closest_indices:
    ax2.plot(np.arange(timesteps) * 10, true_training_solutions[idx, :], color='gray', alpha=0.4, lw=1)
ax2.set_ylabel("Integrated VOD [m]", color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

fig.suptitle("Absolute Difference: Integrated VOD")
fig.tight_layout()
plt.show()


### Add Violin Plots for VOD, Contrail Width, Ice Number, Ice Mass, Median Ice Crystal Radius, Average Crystal Location

In [ ]:
# Violin plot of training outputs as a function of time
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
parts = ax.violinplot(true_training_solutions, showmeans=True, showmedians=True)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Integrated VOD [m]")
ax.set_title(f"APCEMM Predicted Vertical Optical Depth for Mean RHi = 123% and Predicted Uncertainty = {test_specifications.IC_std_rhi*3:.2f}%")
plt.tight_layout()

# Set x-tick labels to be time in minutes (timestep * 10)
xticks = np.arange(0, timesteps, 10)
ax.set_xticks(xticks)
ax.set_xticklabels((xticks * 10).astype(int))

# Add zoomed-in inset for Timestep 7
timestep_idx = 7  # Python index for Timestep 7 (8th violin, since index starts at 0)
data_zoom = true_training_solutions[:, timestep_idx]

axins = inset_axes(ax, width="20%", height="40%", loc='upper right', borderpad=2)
vp = axins.violinplot([data_zoom], showmeans=True, showmedians=True)
axins.set_xticks([1])
axins.set_xticklabels([f"Time [min] {timestep_idx*10}"])
axins.set_ylabel("VOD [m]", fontsize=8)
axins.tick_params(axis='both', which='major', labelsize=8)
axins.set_title('Zoom: Minute 70', fontsize=10)

# Label mean and median
mean_val = data_zoom.mean()
median_val = np.median(data_zoom)
axins.scatter([1], [mean_val], color='blue', zorder=3)
axins.text(0.8, mean_val, 'Mean', color='blue', va='center', fontsize=8)
axins.scatter([1], [median_val], color='red', zorder=3)
axins.text(1.13, median_val, 'Median', color='red', va='center', fontsize=8)

# Set the outline of the inset box to red
for spine in axins.spines.values():
    spine.set_edgecolor('red')
    spine.set_linewidth(2)

# Draw a box on the main plot to show the zoomed region
ymin, ymax = data_zoom.min(), data_zoom.max()
rect = Rectangle((timestep_idx+1-0.5, ymin), 1, ymax-ymin, linewidth=1, edgecolor='red', facecolor='none')
ax.add_patch(rect)

plt.show()

In [ ]:
# Violin plot of selected training outputs (closest_indices) as a function of time
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
selected_solutions = true_training_solutions[closest_indices, :]
parts = ax.violinplot(selected_solutions, showmeans=True, showmedians=True)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Integrated VOD [m]")
ax.set_title("APCEMM Predicted Vertical Optical Depth for Selected RHi Values")
plt.tight_layout()

# Set x-tick labels to be time in minutes (timestep * 10)
xticks = np.arange(0, timesteps, 10)
ax.set_xticks(xticks)
ax.set_xticklabels((xticks * 10).astype(int))

# Add zoomed-in inset for Timestep 7
timestep_idx = 7  # 8th violin, index starts at 0
data_zoom = selected_solutions[:, timestep_idx]

axins = inset_axes(ax, width="20%", height="40%", loc='upper right', borderpad=2)
vp = axins.violinplot([data_zoom], showmeans=True, showmedians=True)
axins.set_xticks([1])
axins.set_xticklabels([f"Time [min] {timestep_idx*10}"])
axins.set_ylabel("VOD [m]", fontsize=8)
axins.tick_params(axis='both', which='major', labelsize=8)
axins.set_title('Zoom: Minute 70', fontsize=10)

# Label mean and median
mean_val = data_zoom.mean()
median_val = np.median(data_zoom)
axins.scatter([1], [mean_val], color='blue', zorder=3)
axins.text(0.8, mean_val, 'Mean', color='blue', va='center', fontsize=8)
axins.scatter([1], [median_val], color='red', zorder=3)
axins.text(1.13, median_val, 'Median', color='red', va='center', fontsize=8)

# Set the outline of the inset box to red
for spine in axins.spines.values():
    spine.set_edgecolor('red')
    spine.set_linewidth(2)

# Draw a box on the main plot to show the zoomed region
ymin, ymax = data_zoom.min(), data_zoom.max()
rect = Rectangle((timestep_idx+1-0.5, ymin), 1, ymax-ymin, linewidth=1, edgecolor='red', facecolor='none')
ax.add_patch(rect)

plt.show()

In [ ]:
# Violin plot of training outputs as a function of time
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
parts = ax.violinplot(true_training_solutions, showmeans=True, showmedians=True)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Integrated VOD [m]")
ax.set_title(f"APCEMM Predicted Vertical Optical Depth for Mean RHi = 123% and Predicted Uncertainty = {test_specifications.IC_std_rhi*3:.2f}%")
plt.tight_layout()

# Set x-tick labels to be time in minutes (timestep * 10)
xticks = np.arange(0, timesteps, 10)
ax.set_xticks(xticks)
ax.set_xticklabels((xticks * 10).astype(int))

# Add zoomed-in inset for Timestep 7
timestep_idx = 60  # Python index for Timestep 7 (8th violin, since index starts at 0)
data_zoom = true_training_solutions[:, timestep_idx]

axins = inset_axes(ax, width="20%", height="40%", loc='upper right', borderpad=2)
vp = axins.violinplot([data_zoom], showmeans=True, showmedians=True)
axins.set_xticks([1])
axins.set_xticklabels([f"Time [min] {timestep_idx*10}"])
axins.set_ylabel("VOD [m]", fontsize=8)
axins.tick_params(axis='both', which='major', labelsize=8)
axins.set_title('Zoom: Hour 10', fontsize=10)

# Label mean and median
mean_val = data_zoom.mean()
median_val = np.median(data_zoom)
axins.scatter([1], [mean_val], color='blue', zorder=3)
axins.text(0.8, mean_val, 'Mean', color='blue', va='center', fontsize=8)
axins.scatter([1], [median_val], color='red', zorder=3)
axins.text(1.13, median_val, 'Median', color='red', va='center', fontsize=8)

# Set the outline of the inset box to red
for spine in axins.spines.values():
    spine.set_edgecolor('red')
    spine.set_linewidth(2)

# Draw a box on the main plot to show the zoomed region
ymin, ymax = data_zoom.min(), data_zoom.max()
rect = Rectangle((timestep_idx+1-0.5, ymin), 1, ymax-ymin, linewidth=1, edgecolor='red', facecolor='none')
ax.add_patch(rect)

plt.show()

In [ ]:
# Print the relative humidity values (RHi) for all runs at timestep 60, sorted by their associated VOD at timestep 60 (high to low)
rhi_timestep_60 = training_inputs[:, 60]
vod_timestep_60 = true_training_solutions[:, 60]
sorted_indices = np.argsort(vod_timestep_60)[::-1]
sorted_rhi_by_vod = rhi_timestep_60[sorted_indices]
sorted_vod = vod_timestep_60[sorted_indices]
print("RHi values at timestep 60 sorted by VOD (high to low):", sorted_rhi_by_vod)
print("Corresponding VOD values:", sorted_vod)

### Import APCEMM Properties not Reported by Surrogate

In [ ]:
IWC_avg = np.zeros((training_runs, timesteps))
effRadius_avg = np.zeros((training_runs, timesteps))
contrailWidth = np.zeros((training_runs, timesteps))
contrailDepth = np.zeros((training_runs, timesteps))

for i in range(training_runs):  # For test runs num_runs
        apcemm_data = lib.read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{i+1}')
        ds_t = apcemm_data.ds_t

        for j in range(timesteps):  # For each timestep
                IWC_avg[i, :] = np.median(ds_t[j]['IWC'])
                effRadius_avg[i, :] = np.median(ds_t[j]['Effective radius'])
                contrailWidth[i, j] = ds_t[j]['width']
                contrailDepth[i, j] = ds_t[j]['depth']

### Plot Contrail Width

In [ ]:
# Smooth out APCEMM anomalies
smoothed_contrailWidth = np.zeros((training_runs, timesteps))
for idx in range(training_runs):
    # Smooth the contrailWidth data for each run
    smoothed_contrailWidth[idx,:] = lib.smooth_artifacts(contrailWidth[idx,:])

In [ ]:
# Violin plot of contrail width as a function of time
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
parts = ax.violinplot(smoothed_contrailWidth, showmeans=True, showmedians=True)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Contrail Width [m]")
ax.set_title("APCEMM Predicted Contrail Width for All Training Runs")
plt.tight_layout()

# Set x-tick labels to be time in minutes (timestep * 10)
xticks = np.arange(0, timesteps, 10)
ax.set_xticks(xticks)
ax.set_xticklabels((xticks * 10).astype(int))

# Add zoomed-in inset for Timestep 7
timestep_idx = 7  # 8th violin, index starts at 0
data_zoom = smoothed_contrailWidth[:, timestep_idx]

axins = inset_axes(ax, width="20%", height="40%", loc="upper left", bbox_to_anchor=(0.1,0,1,1), bbox_transform=ax.transAxes, borderpad=2)
vp = axins.violinplot([data_zoom], showmeans=True, showmedians=True)
axins.set_xticks([1])
axins.set_xticklabels([f"Time [min] {timestep_idx*10}"])
axins.set_ylabel("Width [m]", fontsize=8)
axins.tick_params(axis='both', which='major', labelsize=8)
axins.set_title('Zoom: Minute 70', fontsize=10)

# Label mean and median
mean_val = data_zoom.mean()
median_val = np.median(data_zoom)
axins.scatter([1], [mean_val], color='blue', zorder=3)
axins.text(0.8, mean_val, 'Mean', color='blue', va='center', fontsize=8)
axins.scatter([1], [median_val], color='red', zorder=3)
axins.text(1.13, median_val, 'Median', color='red', va='center', fontsize=8)

# Set the outline of the inset box to red
for spine in axins.spines.values():
    spine.set_edgecolor('red')
    spine.set_linewidth(2)

# Draw a box on the main plot to show the zoomed region
ymin, ymax = data_zoom.min(), data_zoom.max()
rect = Rectangle((timestep_idx+1-0.5, ymin), 1, ymax-ymin, linewidth=1, edgecolor='red', facecolor='none')
ax.add_patch(rect)

plt.show()

In [ ]:
# Plot contrail width as a function of time for the closest_indices runs
plt.figure(figsize=(10, 6), dpi=150)
for idx in closest_indices:
    plt.plot(
        np.arange(timesteps) * 10,
        smoothed_contrailWidth[idx, :],
        label=f"RHi = {rhi_values[idx]:.2f}"
    )
plt.xlabel("Time [min]")
plt.ylabel("Contrail Width [m]")
plt.title("Contrail Width vs Time for Selected RHi Values")
plt.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

### Plot Contrail Depth

In [ ]:
# Smooth out APCEMM anomalies
smoothed_contrailDepth = np.zeros((training_runs, timesteps))
for idx in range(training_runs):
    # Smooth the contrailDepth data for each run
    smoothed_contrailDepth[idx,:] = lib.smooth_artifacts(contrailDepth[idx,:])

In [ ]:
# Violin plot of contrail depth as a function of time
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
parts = ax.violinplot(smoothed_contrailDepth, showmeans=True, showmedians=True)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Contrail Depth [m]")
ax.set_title("APCEMM Predicted Contrail Depth for All Training Runs")
plt.tight_layout()

# Set x-tick labels to be time in minutes (timestep * 10)
xticks = np.arange(0, timesteps, 10)
ax.set_xticks(xticks)
ax.set_xticklabels((xticks * 10).astype(int))

# Add zoomed-in inset for Timestep 7
timestep_idx = 7  # 8th violin, index starts at 0
data_zoom = contrailDepth[:, timestep_idx]

axins = inset_axes(ax, width="20%", height="40%", loc="upper left", bbox_to_anchor=(0.1,0,1,1), bbox_transform=ax.transAxes, borderpad=2)
vp = axins.violinplot([data_zoom], showmeans=True, showmedians=True)
axins.set_xticks([1])
axins.set_xticklabels([f"Time [min] {timestep_idx*10}"])
axins.set_ylabel("Depth [m]", fontsize=8)
axins.tick_params(axis='both', which='major', labelsize=8)
axins.set_title('Zoom: Minute 70', fontsize=10)

# Label mean and median
mean_val = data_zoom.mean()
median_val = np.median(data_zoom)
axins.scatter([1], [mean_val], color='blue', zorder=3)
axins.text(0.8, mean_val, 'Mean', color='blue', va='center', fontsize=8)
axins.scatter([1], [median_val], color='red', zorder=3)
axins.text(1.13, median_val, 'Median', color='red', va='center', fontsize=8)

# Set the outline of the inset box to red
for spine in axins.spines.values():
    spine.set_edgecolor('red')
    spine.set_linewidth(2)

# Draw a box on the main plot to show the zoomed region
ymin, ymax = data_zoom.min(), data_zoom.max()
rect = Rectangle((timestep_idx+1-0.5, ymin), 1, ymax-ymin, linewidth=1, edgecolor='red', facecolor='none')
ax.add_patch(rect)

plt.show()

In [ ]:
# Plot contrail depth as a function of time for the closest_indices runs
plt.figure(figsize=(10, 6), dpi=150)
for idx in closest_indices:
    plt.plot(
        np.arange(timesteps) * 10,
        smoothed_contrailDepth[idx, :],
        label=f"RHi = {rhi_values[idx]:.2f}"
    )
plt.xlabel("Time [min]")
plt.ylabel("Contrail Depth [m]")
plt.title("Contrail Depth vs Time for Selected RHi Values")
plt.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

### Plot IWC

In [ ]:
input_rhis = [140, 130, 120, 110, 100]
closest_indices = [np.abs(np.array(rhi_values) - rhi).argmin() for rhi in input_rhis]
times = [1, 6, 18, 30]
n_rows = len(input_rhis)
n_cols = len(times)

# Step 1: Compute vmin and vmax for each time column across all runs
iwc_min_max = {t: [np.inf, -np.inf] for t in times}

# First pass to collect min and max
for run_idx in closest_indices:
    run = run_idx + 1
    apcemm_data = lib.read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{run}')
    ds_t = apcemm_data.ds_t
    for t in times:
        iwc = ds_t[t]["IWC"]
        iwc_min_max[t][0] = min(iwc_min_max[t][0], iwc.min())
        iwc_min_max[t][1] = max(iwc_min_max[t][1], iwc.max())

# Step 2: Plot with shared vmin/vmax per column
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 2.5 * n_rows), dpi=200)
fig.suptitle("Ice Water Content for Selected RHi Runs", fontsize=16)

contour_sets = [[] for _ in range(n_cols)]

for i, run_idx in enumerate(closest_indices):
    run = run_idx + 1
    apcemm_data = lib.read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{run}')
    ds_t = apcemm_data.ds_t

    for j, t in enumerate(times):
        ds_t_time = ds_t[t]
        X_map, Y_map = np.meshgrid(ds_t_time["x"], ds_t_time["y"])
        ax = axes[i, j] if n_rows > 1 else axes[j]

        vmin, vmax = iwc_min_max[t]
        cf = ax.contourf(X_map, Y_map, ds_t_time["IWC"], vmin=vmin, vmax=vmax)
        contour_sets[j].append(cf)

        ax.set_title(f'Run {run}, t = {int(t/6)}h')

    rhi_val = rhi_values[run_idx]
    axes[i, 0].set_ylabel(f'RHi = {rhi_val:.2f}%', fontsize=12)

# Step 3: Add horizontal colorbars above each column

# Adjust layout to make space and avoid squishing
fig.subplots_adjust(top=0.92)  # Make space for top colorbars

cbar_height = 0.02
cbar_padding = 0.01

for j, t in enumerate(times):
    col_axes = [axes[i, j] for i in range(n_rows)]
    # Get the position of the top-most axis in this column
    bbox = col_axes[0].get_position()
    cbar_ax = fig.add_axes([
        bbox.x0,           # x position
        bbox.y1 + cbar_padding,  # y position just above top axis
        bbox.width,        # same width as subplot
        cbar_height        # small height
    ])
    fig.colorbar(
        contour_sets[j][0],  # just one of the contour sets (they share vmin/vmax)
        cax=cbar_ax,
        orientation='horizontal',
        label=f'IWC (t = {int(t/6)}h)'
    )

fig.tight_layout(rect=[0, 0, 1, 0.88])  # Leave room at top for colorbars and title
fig.subplots_adjust(hspace=0.4) # Vertical space between rows

plt.show()


### Plot Effective Radius Distributions

In [ ]:
input_rhis = [140, 130, 120, 110, 100]
closest_indices = [np.abs(np.array(rhi_values) - rhi).argmin() for rhi in input_rhis]
times = [1, 6, 18, 60]
time_labels = {1: "t = 10 min", 6: "t = 1h", 18: "t = 3h", 60: "t = 10h"}

cmap = cm.get_cmap('Set1', len(input_rhis))

fig, axes = plt.subplots(1, len(times), figsize=(20, 5), dpi=120, sharey=True)

for ax, t in zip(axes, times):
    for run_idx in closest_indices:
        run = run_idx + 1
        apcemm_data = lib.read_apcemm_data(
            f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{run}'
        )
        ds_t = apcemm_data.ds_t
        r = ds_t[t]['r'] * 1e6  # Convert to microns
        y = removeLow(ds_t[t]["Overall size distribution"], cutoff=1e9)
        ax.plot(r, y, label=f"RHi={rhi_values[run_idx]:.2f}%")
    ax.set_xlabel('Particle Radius [μm]')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(time_labels[t])
    if ax == axes[0]:
        ax.set_ylabel('Number Ice Crystals')
    else:
        ax.set_ylabel('')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


### Plot Effective Radius

In [ ]:
input_rhis = [140, 130, 120, 110, 100]
closest_indices = [np.abs(np.array(rhi_values) - rhi).argmin() for rhi in input_rhis]
times = [1, 6, 18, 30]
n_rows = len(input_rhis)
n_cols = len(times)

# Step 1: Compute vmin and vmax for each time column across all runs
effrad_min_max = {t: [np.inf, -np.inf] for t in times}

# First pass to collect min and max
for run_idx in closest_indices:
    run = run_idx + 1
    apcemm_data = lib.read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{run}')
    ds_t = apcemm_data.ds_t
    for t in times:
        effrad = ds_t[t]["Effective radius"]
        effrad_min_max[t][0] = min(effrad_min_max[t][0], effrad.min())
        effrad_min_max[t][1] = max(effrad_min_max[t][1], effrad.max())

print("Effective Radius Min/Max Values:")
for t in times:
    print(f"t = {int(t/6)}h: Min = {float(effrad_min_max[t][0])}, Max = {float(effrad_min_max[t][1])}")

# Step 2: Plot with shared vmin/vmax per column
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 2.5 * n_rows), dpi=200)
fig.suptitle("Effective Radius for Selected RHi Runs", fontsize=16)

contour_sets = [[] for _ in range(n_cols)]

for i, run_idx in enumerate(closest_indices):
    run = run_idx + 1
    apcemm_data = lib.read_apcemm_data(f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{run}')
    ds_t = apcemm_data.ds_t

    for j, t in enumerate(times):
        ds_t_time = ds_t[t]
        X_map, Y_map = np.meshgrid(ds_t_time["x"], ds_t_time["y"])
        ax = axes[i, j] if n_rows > 1 else axes[j]

        vmin, vmax = effrad_min_max[t]
        cf = ax.contourf(X_map, Y_map, ds_t_time["Effective radius"], vmin=vmin, vmax=vmax)
        contour_sets[j].append(cf)

        ax.set_title(f'Run {run}, t = {int(t/6)}h')

    rhi_val = rhi_values[run_idx]
    axes[i, 0].set_ylabel(f'RHi = {rhi_val:.2f}%', fontsize=12)

# Step 3: Add horizontal colorbars above each column

fig.subplots_adjust(top=0.92)  # Make space for top colorbars

cbar_height = 0.02
cbar_padding = 0.01

for j, t in enumerate(times):
    col_axes = [axes[i, j] for i in range(n_rows)]
    bbox = col_axes[0].get_position()
    cbar_ax = fig.add_axes([
        bbox.x0,
        bbox.y1 + cbar_padding,
        bbox.width,
        cbar_height
    ])
    fig.colorbar(
        contour_sets[j][-1],
        cax=cbar_ax,
        orientation='horizontal',
        label='R$_{eff}$ (t = '+f'{int(t/6)}h)'
    )

fig.tight_layout(rect=[0, 0, 1, 0.88])
fig.subplots_adjust(hspace=0.4)

plt.show()

In [ ]:
# Plot the effective radius as a function of time for run 19 (Python index 18)
run_idx = 18  # run 19 (0-based index)
run = run_idx + 1

apcemm_data = lib.read_apcemm_data(
    f'/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_data_sets/{test_id}/outputs/training/{test_id}_run_{run}'
)
ds_t = apcemm_data.ds_t

times = [1, 6, 18, 30]
time_labels = {1: "t = 10 min", 6: "t = 1h", 18: "t = 3h", 30: "t = 5h"}

# Find global vmin and vmax for effective radius across all selected times
all_effrad = [ds_t[t]["Effective radius"].values * 10**6 for t in times]
vmin = min([arr.min() for arr in all_effrad])
vmax = max([arr.max() for arr in all_effrad])

fig, axes = plt.subplots(1, len(times), figsize=(20, 5), dpi=120)
plt.subplots_adjust(right=0.85)  # Make space for colorbar

contourf_list = []
for ax, t in zip(axes, times):
    ds_t_time = ds_t[t]
    X_map, Y_map = np.meshgrid(ds_t_time["x"], ds_t_time["y"])
    effrad = ds_t_time["Effective radius"] * 10**6
    cf = ax.contourf(X_map, Y_map, effrad, vmin=vmin, vmax=vmax)
    contourf_list.append(cf)
    ax.set_title(time_labels[t])
    ax.set_xlabel("x")
    if ax == axes[0]:
        ax.set_ylabel("y")

# Add a single colorbar to the right of the figure
cbar_ax = fig.add_axes([0.86, 0.15, 0.02, 0.7])
fig.colorbar(contourf_list[-1], cax=cbar_ax, orientation='vertical', label=r'Effective Radius [$\mu$m]')

plt.suptitle(f"Effective Radius for RHi {rhi_values[run_idx]:.2f}% at Selected Times", fontsize=16)
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()

### Plot the Ice Particle Number

### Add Plot of Contrail Lifetime with IC

### Plot Ice Mass vs Time Overlay of all Training Data

### Plot 2D slice over time (video) for multiple runs

## Plot the validation runs

In [ ]:
# Plot all validation outputs and APCEMM solutions on the same figure
plt.figure(figsize=(10, 6), dpi=150)
for idx in range(validation_runs):
    # Use a unique color for each run so both lines match
    color = f"C{idx}"
    plt.plot(range(timesteps), true_validation_solutions[idx, :], label=f"True Run {idx+1}", lw=1.5, color=color)
    plt.plot(range(timesteps), predicted_validation_solutions[idx, :], label=f"Predicted Run {idx+1}", lw=1.5, linestyle='--', color=color)

plt.xlabel("Timestep")
plt.ylabel("Integrated VOD [m]")
plt.title("Training APCEMM Solutions")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(validation_runs, 1, figsize=(10, 4 * validation_runs), dpi=150, sharex=True)

for idx in range(validation_runs):
    ax = axes[idx]
    color = f"C{idx}"
    ax.plot(range(timesteps), true_validation_solutions[idx, :], label=f"True Run {idx+1}", lw=1.5, color=color)
    ax.plot(range(timesteps), predicted_validation_solutions[idx, :], label=f"Predicted Run {idx+1}", lw=1.5, linestyle='--', color=color)
    ax.set_ylabel("Integrated VOD [m]")
    ax.set_title(f"Validation Run: {validation_inputs[idx, 0]:.2f}% RHi")
    ax.legend(fontsize=9)

axes[-1].set_xlabel("Timestep")
plt.tight_layout()
plt.show()

## Plot novel input results

In [ ]:
plt.figure(figsize=(10, 6), dpi=150)
lines = []
labels = []
rhi_plot = []

cmap = cm.get_cmap('tab20', num_lines)

line_idx = 0
for idx in range(novel_runs):
    color = cmap(line_idx)
    line, = plt.plot(
        np.arange(timesteps) * 10,  # Multiply x axis by 10
        novel_solutions[idx, :],
        label=f"RHi = {novel_inputs[idx,0]:.2f}",
        lw=1.5,
        color=color
    )
    lines.append(line)
    labels.append(f"RHi = {novel_inputs[idx,0]:.2f}")
    rhi_plot.append(novel_inputs[idx,0])
    line_idx += 1

sorted_indices = np.argsort(rhi_plot)
sorted_lines = [lines[i] for i in sorted_indices]
sorted_labels = [labels[i] for i in sorted_indices]

plt.xlabel("Time [min]")
plt.ylabel("Integrated VOD [m]")
plt.title("PCE Predicted Vertical Optical Depth for Mean RHi = 123% and Predicted Uncertainty = 31.57%")
plt.legend(sorted_lines, sorted_labels, fontsize=9)

plt.tight_layout()
plt.show()


## Do the same analysis for MLD